In [5]:
#!/usr/bin/env python3
"""
Surrogate decision-tree distillation — corrected for the actual DDQN training setup.

Key fixes vs. the original script:
  1. Multi-episode rollout instead of one long episode → richer state diversity
  2. Phase-transition logic now matches the training env exactly
     (action==1 AND time_in_phase>=MIN_GREEN_TIME → advance phase; else blocked)
  3. Stratified train/test split on action label (handles class imbalance)
  4. Action-distribution sanity check before fitting trees
  5. Feature-importance report to detect policy collapse
  6. FEATURE_NAMES matched exactly to get_intersection_state() output order
"""

import os
import numpy as np
import tensorflow as tf
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import traci
from tf_agents.trajectories import time_step as ts

# ── CONFIG — must match training constants exactly ────────────────────────────
POLICY_PATH = r"C:\Users\Dell\Desktop\Project\Implementation\Sim1\policy_checkpoints_phase2\best_policy"
SUMO_CFG    = r"C:\Users\Dell\Desktop\Project\Implementation\Sim1\SUMO_FILES\sim.sumocfg"
ADDITIONAL  = r"C:\Users\Dell\Desktop\Project\Implementation\Sim1\SUMO_FILES\sim.add.xml"
TLS_ID      = "Node2"

DETECTORS = [
    "Node1_2_EB_0", "Node1_2_EB_1", "Node1_2_EB_2",
    "Node2_7_SB_0", "Node2_7_SB_1", "Node2_7_SB_2",
]

# Must match training env
MIN_GREEN_TIME = 25
ACTIONS        = 2      # binary: stay(0) / switch(1)
NUM_PHASES     = 4      # queried at runtime but set here for feature naming

OUT_DIR = r"C:\Users\Dell\Desktop\Project\Implementation\Sim1\tree_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ── FEATURE NAMES — must mirror get_intersection_state() exactly ──────────────
# Order: [queue, occ, speed, mean_wait, ev_count] × 6 detectors
#        + [time_since_switch]
#        + phase one-hot × NUM_PHASES
#        + [pressure, demand]
FEATURE_NAMES = []
for det in DETECTORS:
    for feat in ["queue", "occ", "speed", "mean_wait", "ev_count"]:
        FEATURE_NAMES.append(f"{det}_{feat}")
FEATURE_NAMES.append("time_since_switch")
for p in range(NUM_PHASES):
    FEATURE_NAMES.append(f"phase_{p}")
FEATURE_NAMES += ["pressure", "demand"]

# Normalization constants — must match training env
MAX_QUEUE, MAX_OCC, MAX_SPEED, MAX_WAIT, MAX_EV = 50., 100., 20., 300., 5.
MAX_TIME_IN = 300.


# ── DETECTOR + STATE HELPERS (copied from training env) ──────────────────────

def gather_detector_data():
    data = []
    for det in DETECTORS:
        veh_ids   = traci.lanearea.getLastStepVehicleIDs(det)
        queue     = traci.lanearea.getJamLengthVehicle(det)
        occ       = traci.lanearea.getLastStepOccupancy(det)
        speed     = traci.lanearea.getLastStepMeanSpeed(det)
        veh_count = traci.lanearea.getLastStepVehicleNumber(det)

        ev_count, ev_waiting, wait_times = 0., 0, []
        for vid in veh_ids:
            wait_times.append(traci.vehicle.getWaitingTime(vid))
            if traci.vehicle.getVehicleClass(vid) == 'emergency':
                ev_count += 1.
                if traci.vehicle.getSpeed(vid) < 1.0:
                    ev_waiting += 1

        mean_wait = float(np.mean(wait_times)) if wait_times else 0.
        data.append(dict(queue=queue, occ=occ, speed=speed, veh_count=veh_count,
                         mean_wait=mean_wait, ev_count=ev_count, ev_waiting=ev_waiting))
    return data


def get_state(det_data, current_phase, time_since_switch, num_phases):
    """Mirrors get_intersection_state() from training env."""
    features = []
    for d in det_data:
        features.extend([d["queue"], d["occ"], d["speed"], d["mean_wait"], d["ev_count"]])
    features.append(float(time_since_switch))

    max_vals = np.array(
        [MAX_QUEUE, MAX_OCC, MAX_SPEED, MAX_WAIT, MAX_EV] * len(DETECTORS) + [MAX_TIME_IN],
        dtype=np.float32,
    )
    continuous  = np.array(features, dtype=np.float32)
    normalized  = np.clip(continuous / (max_vals + 1e-5), 0., 1.)

    total_queue = sum(d["queue"] for d in det_data)
    pressure    = np.clip(total_queue / (MAX_QUEUE * len(DETECTORS)), 0., 1.)
    total_veh   = sum(d["veh_count"] for d in det_data)
    demand      = np.clip(total_veh / (10. * len(DETECTORS)), 0., 1.)

    phase_oh = np.zeros(num_phases, dtype=np.float32)
    if 0 <= current_phase < num_phases:
        phase_oh[current_phase] = 1.

    return np.concatenate([normalized, phase_oh, [pressure, demand]]).astype(np.float32)


# ── ROLLOUT ───────────────────────────────────────────────────────────────────

def collect_episode(policy, max_steps, seed=42):
    """
    Run one short episode and return (states, actions).

    FIX: transition logic now matches the training env exactly:
      - action==1 AND time_in_phase>=MIN_GREEN_TIME → advance phase
      - action==1 but time_in_phase<MIN_GREEN_TIME  → blocked, phase unchanged
      - action==0                                   → stay
    The original script only advanced on action==1, missing the MIN_GREEN_TIME
    guard, so collected states were off-distribution.
    """
    sumo_cmd = [
        "sumo", "-c", SUMO_CFG,
        "--route-files",
        r"C:\Users\Dell\Desktop\Project\Implementation\Sim1\SUMO_FILES\traffic_files\ev.rou.xml,"
        r"C:\Users\Dell\Desktop\Project\Implementation\Sim1\SUMO_FILES\traffic_files\passenger.rou.xml",
        "--additional-files", ADDITIONAL,
        "--no-step-log", "--no-warnings", "--seed", str(seed),
    ]
    traci.start(sumo_cmd)

    states, actions = [], []
    try:
        num_phases    = len(traci.trafficlight.getAllProgramLogics(TLS_ID)[0].phases)
        current_phase = 0
        time_in_phase = 0

        for step in range(max_steps):
            det_data  = gather_detector_data()
            state_vec = get_state(det_data, current_phase, time_in_phase, num_phases)

            obs_t     = tf.expand_dims(tf.constant(state_vec, dtype=tf.float32), 0)
            time_step = ts.TimeStep(
                step_type   = tf.constant([ts.StepType.MID],  dtype=tf.int32),
                reward      = tf.constant([0.0],              dtype=tf.float32),
                discount    = tf.constant([1.0],              dtype=tf.float32),
                observation = obs_t,
            )
            action_step = policy.action(time_step)
            action      = int(action_step.action.numpy().flat[0])

            states.append(state_vec)
            actions.append(action)

            # ── FIX: replicate training env transition logic exactly ──
            if action == 1 and time_in_phase >= MIN_GREEN_TIME:
                current_phase = (current_phase + 1) % num_phases
                traci.trafficlight.setPhase(TLS_ID, current_phase)
                time_in_phase = 0
            elif action == 1:
                # blocked — phase unchanged, timer still ticks
                time_in_phase += 1
            else:
                time_in_phase += 1

            traci.simulationStep()

            if traci.simulation.getMinExpectedNumber() <= 0:
                break
    finally:
        traci.close()

    return np.array(states), np.array(actions)


def collect_multi_episode(policy, n_episodes=10, steps_per_ep=600):
    """
    FIX: Run multiple short episodes with different seeds.
    One long episode under-samples the early-episode state distribution
    (no traffic yet) and over-samples steady-state. Diversity matters for
    surrogate fidelity — especially for the minority action_1 class.
    """
    all_states, all_actions = [], []
    for ep in range(n_episodes):
        seed = 42 + ep * 7
        print(f"  Episode {ep+1}/{n_episodes} (seed={seed}) ...", end=" ", flush=True)
        s, a = collect_episode(policy, max_steps=steps_per_ep, seed=seed)
        all_states.append(s)
        all_actions.append(a)
        rate = a.mean() * 100
        print(f"{len(s)} steps, action_1 rate={rate:.1f}%")

    return np.vstack(all_states), np.concatenate(all_actions)


# ── SANITY CHECK ──────────────────────────────────────────────────────────────

def check_policy_health(y):
    """
    FIX: Abort early if action_1 rate is nearly zero.
    The original script would silently fit a tree that predicts action_0
    for everything and report ~97% fidelity — which is misleading because
    a trivial majority-class classifier gets the same score.
    """
    rate = y.mean()
    print(f"\nAction distribution: action_0={1-rate:.1%}, action_1={rate:.1%}")
    if rate < 0.02:
        print(
            "\n[WARNING] action_1 rate < 2%. The policy is almost certainly\n"
            "collapsed to always staying. The surrogate tree will be trivially\n"
            "action_0 and tells you nothing useful about the policy's decision\n"
            "boundary. Diagnose the training run before proceeding.\n"
            "\nSigns to look for in training_metrics.csv:\n"
            "  • act1_counts column near zero from episode ~50 onward\n"
            "  • Q-values for action_1 collapsed below action_0 after initial exploration\n"
            "  • Reward plateau well below zero (queue/wait penalties dominate)\n"
        )
        return False
    return True


# ── TREE FITTING ──────────────────────────────────────────────────────────────

def fit_surrogate(X, y):
    """
    FIX: Stratified split — plain random split can exclude action_1 from
    the test set entirely when it's a tiny minority, making fidelity look
    artificially high.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    best_tree, best_depth, best_acc = None, None, 0.

    for depth in [2, 3, 4, 5]:
        tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
        tree.fit(X_train, y_train)
        acc = tree.score(X_test, y_test)
        print(f"  Depth {depth}: fidelity={acc:.3f}")

        if acc > 0.85 and best_tree is None:
            best_tree  = tree
            best_depth = depth
            best_acc   = acc

    if best_tree is None:
        best_depth = 3
        best_tree  = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
        best_tree.fit(X_train, y_train)
        best_acc = best_tree.score(X_test, y_test)
        print(f"  No depth crossed 85%. Using depth={best_depth}, fidelity={best_acc:.3f}")

    print(f"\nFinal surrogate: depth={best_depth}, fidelity={best_acc:.3f}")
    return best_tree


# ── FEATURE IMPORTANCE REPORT ─────────────────────────────────────────────────

def report_feature_importance(tree, feature_names, top_n=10):
    """
    FIX: Print top features. If 'demand' alone dominates (>0.8 importance)
    that confirms the tree learned a trivial policy (switch only when traffic
    is present at all), not a genuine phase-selection strategy.
    """
    importances = tree.feature_importances_
    top_idx     = np.argsort(importances)[::-1][:top_n]

    print("\nTop feature importances:")
    for i in top_idx:
        bar = "█" * int(importances[i] * 40)
        print(f"  {feature_names[i]:<35} {importances[i]:.3f}  {bar}")

    top_feat = feature_names[top_idx[0]]
    if top_feat == "demand" and importances[top_idx[0]] > 0.7:
        print(
            "\n[WARNING] 'demand' accounts for >70% of feature importance.\n"
            "This almost always means the policy collapsed to 'switch when\n"
            "any vehicles are present, stay otherwise'. The surrogate tree is\n"
            "faithfully capturing a degenerate policy, not a learned one.\n"
        )


# ── DIAGRAMS ──────────────────────────────────────────────────────────────────

def save_tree_diagram(tree, feature_names, class_names, filepath):
    fig, ax = plt.subplots(figsize=(28, 20))
    plot_tree(
        tree,
        feature_names=feature_names,
        class_names=class_names,
        filled=True, rounded=True,
        fontsize=9, proportion=True, precision=2, ax=ax,
    )
    ax.set_title("Surrogate Decision Tree — Extracted from DDQN Policy", fontsize=18, pad=20)
    plt.tight_layout()
    fig.savefig(filepath, dpi=300, bbox_inches="tight", format="png")
    plt.close(fig)
    print(f"Matplotlib PNG: {filepath}")


def save_tree_graphviz(tree, feature_names, class_names, filepath):
    try:
        from sklearn.tree import export_graphviz
        import graphviz
        dot = export_graphviz(tree, out_file=None, feature_names=feature_names,
                              class_names=class_names, filled=True, rounded=True,
                              special_characters=True)
        graphviz.Source(dot).render(filepath, format="png", cleanup=True)
        print(f"Graphviz PNG: {filepath}.png")
    except Exception as e:
        print(f"Graphviz skipped ({e}). Matplotlib PNG already saved.")


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():
    print(f"Loading policy from: {POLICY_PATH}")
    policy = tf.saved_model.load(POLICY_PATH)

    print("\nCollecting rollouts ...")
    X, y = collect_multi_episode(policy, n_episodes=20, steps_per_ep=5000)
    print(f"Total samples: {len(X)}")

    if not check_policy_health(y):
        return   # policy collapsed — no point fitting a tree

    print("\nFitting surrogate trees ...")
    tree = fit_surrogate(X, y)

    report_feature_importance(tree, FEATURE_NAMES)

    # Diagrams
    class_names = [f"action_{a}" for a in range(ACTIONS)]
    png_path = os.path.join(OUT_DIR, "surrogate_tree.png")
    save_tree_diagram(tree, FEATURE_NAMES, class_names, png_path)
    save_tree_graphviz(tree, FEATURE_NAMES, class_names,
                       os.path.join(OUT_DIR, "surrogate_tree_graphviz"))

    # Text rules
    rules_path = os.path.join(OUT_DIR, "surrogate_tree_rules.txt")
    with open(rules_path, "w") as f:
        f.write(export_text(tree, feature_names=FEATURE_NAMES))
    print(f"\nText rules: {rules_path}")

    # Inline print
    from sklearn.tree import _tree
    tree_ = tree.tree_

    def recurse(node, depth=0):
        indent = "  " * depth
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name      = FEATURE_NAMES[tree_.feature[node]]
            threshold = tree_.threshold[node]
            print(f"{indent}if {name} <= {threshold:.3f}:")
            recurse(tree_.children_left[node],  depth + 1)
            print(f"{indent}else:")
            recurse(tree_.children_right[node], depth + 1)
        else:
            counts = tree_.value[node][0]
            pred   = np.argmax(counts)
            pct    = counts[pred] / counts.sum() * 100
            print(f"{indent}→ action={pred}  ({pct:.0f}% pure, n={int(counts.sum())})")

    print("\n--- Surrogate tree rules ---")
    recurse(0)


if __name__ == "__main__":
    main()

Loading policy from: C:\Users\Dell\Desktop\Project\Implementation\Sim1\policy_checkpoints_phase2\best_policy

  Episode 1/20 (seed=42) ... 4323 steps, action_1 rate=2.3%
  Episode 2/20 (seed=49) ... 4293 steps, action_1 rate=2.4%
  Episode 3/20 (seed=56) ... 4303 steps, action_1 rate=1.7%
  Episode 4/20 (seed=63) ... 4289 steps, action_1 rate=2.4%
  Episode 5/20 (seed=70) ... 4263 steps, action_1 rate=2.0%
  Episode 6/20 (seed=77) ... 4303 steps, action_1 rate=2.1%
  Episode 7/20 (seed=84) ... 4281 steps, action_1 rate=3.0%
  Episode 8/20 (seed=91) ... 4322 steps, action_1 rate=3.7%
  Episode 9/20 (seed=98) ... 4301 steps, action_1 rate=2.0%
  Episode 10/20 (seed=105) ... 4273 steps, action_1 rate=2.2%
  Episode 11/20 (seed=112) ... 4311 steps, action_1 rate=3.0%
  Episode 12/20 (seed=119) ... 4318 steps, action_1 rate=2.3%
  Episode 13/20 (seed=126) ... 4303 steps, action_1 rate=1.9%
  Episode 14/20 (seed=133) ... 4292 steps, action_1 rate=2.5%
  Episode 15/20 (seed=140) ... 4279 step

In [ ]:
traci.close()

In [4]:
import shap
import numpy as np
import tensorflow as tf
from tf_agents.trajectories import time_step as ts
import matplotlib.pyplot as plt


# ─────────────────────────── CONFIG (from your training script) ───────────────────────────
DETECTORS = [
    "Node1_2_EB_0", "Node1_2_EB_1", "Node1_2_EB_2",
    "Node2_7_SB_0", "Node2_7_SB_1", "Node2_7_SB_2",
]

ACTIONS = 2
MIN_GREEN_TIME = 25

# Normalization maximums (from your training script)
MAX_QUEUE     = 50.0
MAX_OCCUPANCY = 100.0
MAX_SPEED     = 20.0
MAX_WAIT      = 300.0
MAX_EV        = 5.0
MAX_TIME_IN   = 300.0


# ─────────────────────────── DYNAMIC FEATURE NAME BUILDER ───────────────────────────
def build_feature_names(num_phases: int) -> list[str]:
    """
    Builds feature names that EXACTLY match the state vector produced by
    get_intersection_state(). The order must be identical.
    """
    names = []
    
    # 1. Per-detector features: queue, occ, speed, mean_wait, ev_count
    for det in DETECTORS:
        names.extend([
            f"{det}_queue",
            f"{det}_occ", 
            f"{det}_speed",
            f"{det}_wait",
            f"{det}_ev",
        ])
    
    # 2. Temporal feature
    names.append("time_since_switch")
    
    # 3. Phase one-hot (size = num_phases)
    for i in range(num_phases):
        names.append(f"phase_{i}")
    
    # 4. Summary globals
    names.extend(["pressure", "demand"])
    
    assert len(names) == 5 * len(DETECTORS) + 1 + num_phases + 2
    return names


# ─────────────────────────── SHAP ANALYSIS FUNCTION ───────────────────────────
def shap_analysis(agent, background_states, test_states, num_phases):
    """
    Explain the DDQN agent's switch decisions using SHAP.
    
    Parameters
    ----------
    agent : tf_agents.agents.dqn.dqn_agent.DdqnAgent
        The trained DDQN agent. Uses agent._q_network (greedy eval) for SHAP.
    background_states : np.ndarray, shape (N, state_dim)
        Representative states for the SHAP baseline (e.g., 100 random states).
    test_states : np.ndarray, shape (M, state_dim)
        States to explain (sample from evaluation episodes).
    num_phases : int
        Number of traffic light phases (from SUMO network).
    
    Returns
    -------
    shap_values : np.ndarray, shape (M, state_dim)
        SHAP values for each feature.
    """
    feature_names = build_feature_names(num_phases)
    state_dim = len(feature_names)
    
    # ── Validate dimensions ──
    assert background_states.shape[1] == state_dim, \
        f"Background states dim {background_states.shape[1]} != expected {state_dim}"
    assert test_states.shape[1] == state_dim, \
        f"Test states dim {test_states.shape[1]} != expected {state_dim}"
    
    q_net = agent._q_network
    
    # ── 1. Black-box: continuous, fully-batched P(action=1) ──
    # We use softmax over Q-values instead of discrete argmax.
    # This gives a smooth, differentiable probability that KernelExplainer needs.
    def f(states):
        """
        Returns P(switch | state) via softmax over Q-values.
        Shape: (N, 1) where N is number of input states.
        """
        if len(states.shape) == 1:
            states = states[np.newaxis, :]
        states_tf = tf.constant(states, dtype=tf.float32)
        q_vals, _ = q_net(states_tf)           # (N, ACTIONS)
        probs = tf.nn.softmax(q_vals, axis=-1)   # (N, ACTIONS)
        return probs[:, 1:2].numpy()             # (N, 1) — probability of action=1
    
    # ── Sanity check: verify f matches greedy policy ──
    # Pick one test state and compare Q-network softmax argmax vs policy action
    sample_obs = tf.expand_dims(tf.constant(test_states[0], dtype=tf.float32), 0)
    sample_tstep = ts.transition(
        observation=sample_obs,
        reward=tf.constant([0.0], dtype=tf.float32),
        discount=tf.constant([1.0], dtype=tf.float32),
    )
    policy_action = int(agent.policy.action(sample_tstep).action.numpy().flat[0])
    q_action = int(np.argmax(q_net(sample_obs)[0].numpy()))
    assert policy_action == q_action, \
        f"Policy/Q-network mismatch! Policy says {policy_action}, Q says {q_action}"
    print(f"  [SHAP] Sanity check passed: policy={policy_action}, Q-argmax={q_action}")
    
    # ── 2. KernelExplainer ──
    print(f"  [SHAP] Fitting KernelExplainer with {len(background_states)} background samples...")
    explainer = shap.KernelExplainer(f, background_states)
    
    print(f"  [SHAP] Computing SHAP values for {len(test_states)} test samples (nsamples=100)...")
    shap_values = explainer.shap_values(test_states, nsamples=100)
    sv = shap_values[0]  # Single-output: (M, state_dim)
    
    # ── 3. Global summary plot ──
    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, test_states, feature_names=feature_names, show=False)
    plt.title("SHAP: Feature Influence on Switch Probability")
    plt.tight_layout()
    plt.savefig("shap_summary.png", dpi=200)
    plt.show()
    print("  [SHAP] Saved: shap_summary.png")
    
    # ── 4. Detector heatmap (first 30 features = 6 detectors × 5 features) ──
    n_detectors = len(DETECTORS)
    det_feats = 5  # queue, occ, speed, wait, ev
    
    det_shap = sv[:, :n_detectors * det_feats].reshape(-1, n_detectors, det_feats)
    det_shap_mean = np.abs(det_shap).mean(axis=0)  # (6, 5)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(det_shap_mean, cmap='RdYlGn_r', aspect='auto')
    ax.set_xticks(range(det_feats))
    ax.set_xticklabels(["Queue", "Occ", "Speed", "Wait", "EV"])
    ax.set_yticks(range(n_detectors))
    ax.set_yticklabels(DETECTORS)
    ax.set_title("Mean |SHAP| per Detector × Feature\n(Red = High Influence on Switch Decision)")
    plt.colorbar(im, ax=ax, label="Mean |SHAP value|")
    plt.tight_layout()
    plt.savefig("shap_detector_heatmap.png", dpi=200)
    plt.show()
    print("  [SHAP] Saved: shap_detector_heatmap.png")
    
    # ── 5. Global / non-detector feature importance ──
    global_start = n_detectors * det_feats
    if sv.shape[1] > global_start:
        global_sv = sv[:, global_start:]
        global_names = feature_names[global_start:]
        global_mean = np.abs(global_sv).mean(axis=0)
        
        fig, ax = plt.subplots(figsize=(8, 4))
        y_pos = np.arange(len(global_names))
        ax.barh(y_pos, global_mean, color='steelblue')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(global_names)
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title("Global / Non-Detector Feature Influence")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig("shap_global_features.png", dpi=200)
        plt.show()
        print("  [SHAP] Saved: shap_global_features.png")
    
    return sv


# ─────────────────────────── STATE COLLECTION HELPER ───────────────────────────
def collect_states_for_shap(tf_env, agent, num_background=100, num_test=50, num_phases=None):
    """
    Collect normalized states from evaluation episodes for SHAP analysis.
    
    Run this AFTER training is complete, using the saved policy or loaded agent.
    """
    if num_phases is None:
        # Try to infer from agent's Q-network output dimension
        num_phases = None  # You must set this based on your SUMO network
    
    all_states = []
    
    # Run a few evaluation episodes to collect states
    for _ in range(max(5, (num_background + num_test) // 20)):
        time_step = tf_env.reset()
        while not time_step.is_last():
            action_step = agent.policy.action(time_step)
            obs = time_step.observation.numpy().flatten()
            all_states.append(obs)
            time_step = tf_env.step(action_step.action)
    
    all_states = np.array(all_states)
    np.random.shuffle(all_states)
    
    bg = all_states[:num_background]
    test = all_states[num_background:num_background + num_test]
    
    print(f"  [SHAP] Collected {len(all_states)} states total.")
    print(f"  [SHAP] Background: {bg.shape}, Test: {test.shape}")
    
    return bg, test


# ═══════════════════════════ USAGE EXAMPLE ═══════════════════════════
# 
# After training completes (or after loading a checkpoint):
#
#   # 1. Determine num_phases from your SUMO network
#   #    (e.g., inspect traci.trafficlight.getAllProgramLogics(TLS_ID)[0].phases)
#   num_phases = 4   # <-- SET THIS to match your actual network
#
#   # 2. Collect states
#   bg_states, test_states = collect_states_for_shap(
#       tf_env, agent, num_background=100, num_test=50, num_phases=num_phases
#   )
#
#   # 3. Run SHAP
#   shap_values = shap_analysis(agent, bg_states, test_states, num_phases)
#
# ═══════════════════════════════════════════════════════════════════════

In [ ]:
import os
import numpy as np
import tensorflow as tf
import traci
from tf_agents.environments import tf_py_environment
from tf_agents.trajectories import time_step as ts
import shap
import matplotlib.pyplot as plt
from phase2_horrible import SUMOTrafficEnv, build_agent

# ── Reconstruct everything exactly as in training ──
SUMO_CFG   = "SUMO_FILES/sim.sumocfg"
ADDITIONAL = "SUMO_FILES/sim.add.xml"
TLS_ID     = "Node2"
DETECTORS = [
    "Node1_2_EB_0", "Node1_2_EB_1", "Node1_2_EB_2",
    "Node2_7_SB_0", "Node2_7_SB_1", "Node2_7_SB_2",
]

sumo_cmd = [
    "sumo", "-c", SUMO_CFG,
    "--route-files", "SUMO_FILES/traffic_files/ev.rou.xml,SUMO_FILES/traffic_files/passenger.rou.xml",
    "--additional-files", ADDITIONAL,
    "--no-step-log", "--no-warnings",
]

# 1. Create environment (must match training exactly)
py_env = SUMOTrafficEnv(sumo_cmd)
tf_env = tf_py_environment.TFPyEnvironment(py_env)

# 2. Rebuild agent with dummy variables (structure must match)
global_step  = tf.Variable(0, trainable=False, dtype=tf.int64)
env_step_var = tf.Variable(0, trainable=False, dtype=tf.int64)
epsilon_var  = tf.Variable(0.05, trainable=False, dtype=tf.float32)  # eval epsilon

agent, _ = build_agent(tf_env, global_step, env_step_var, epsilon_var)

# 3. Restore checkpoint
checkpointer = common.Checkpointer(
    ckpt_dir="./checkpoints_phase2",
    agent=agent,
    policy=agent.policy,
    global_step=global_step,
)
checkpointer.initialize_or_restore()
print(f"Restored from step {int(global_step.numpy())}")

# 4. Run SHAP
num_phases = py_env._num_phases
shap_values = run_shap_analysis(agent, tf_env, py_env, num_phases)

tf_env.close()

In [5]:
import xml.etree.ElementTree as ET

def inspect_tls_phases(net_file: str, tls_id: str):
    """Print phase states and controlled lanes from the net file."""
    tree = ET.parse(net_file)
    root = tree.getroot()

    for tl in root.findall(".//tlLogic"):
        if tl.get("id") != tls_id:
            continue
        print(f"TLS: {tl.get('id')}  program: {tl.get('programID')}")
        for i, phase in enumerate(tl.findall("phase")):
            print(f"  Phase {i}: state={phase.get('state')}  "
                  f"duration={phase.get('duration')}")

    # Print which lanes each signal index controls
    for conn in root.findall(".//connection"):
        if conn.get("tl") == tls_id:
            print(f"  Signal {conn.get('linkIndex'):>2} → "
                  f"from={conn.get('from')}_{conn.get('fromLane')}  "
                  f"to={conn.get('to')}_{conn.get('toLane')}")

inspect_tls_phases(r'C:\Users\Dell\Desktop\Project\Implementation\Sim1\SUMO_FILES\sim.net.xml', 'Node2')

TLS: Node2  program: 0
  Phase 0: state=GGGGgrrrrrGGGGgrrrrr  duration=42
  Phase 1: state=yyyyyrrrrryyyyyrrrrr  duration=3
  Phase 2: state=rrrrrGGGGgrrrrrGGGGg  duration=42
  Phase 3: state=rrrrryyyyyrrrrryyyyy  duration=3
  Signal 10 → from=Node1_2_EB_0  to=Node2_5_SB_0
  Signal 11 → from=Node1_2_EB_0  to=Node2_3_EB_0
  Signal 12 → from=Node1_2_EB_1  to=Node2_3_EB_1
  Signal 13 → from=Node1_2_EB_2  to=Node2_3_EB_2
  Signal 14 → from=Node1_2_EB_2  to=Node2_7_NB_2
  Signal  0 → from=Node2_3_WB_0  to=Node2_7_NB_0
  Signal  1 → from=Node2_3_WB_0  to=Node1_2_WB_0
  Signal  2 → from=Node2_3_WB_1  to=Node1_2_WB_1
  Signal  3 → from=Node2_3_WB_2  to=Node1_2_WB_2
  Signal  4 → from=Node2_3_WB_2  to=Node2_5_SB_2
  Signal  5 → from=Node2_5_WB_0  to=Node2_3_EB_0
  Signal  6 → from=Node2_5_WB_0  to=Node2_7_NB_0
  Signal  7 → from=Node2_5_WB_1  to=Node2_7_NB_1
  Signal  8 → from=Node2_5_WB_2  to=Node2_7_NB_2
  Signal  9 → from=Node2_5_WB_2  to=Node1_2_WB_2
  Signal 15 → from=Node2_7_SB_0  to=Node